# Experiment with noisy features

In [18]:
import sys
import os
import torch
sys.path.append(os.path.join(os.getcwd(), 'src')) 
from graphLoader import GraphLoader
import time
from torch_geometric.logging import log
import gc 
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
import pandas as pd
import os
from pyHSICLasso import HSICLasso
from collections import Counter
from tqdm import tqdm 
from torch_geometric.explain import Explainer, GNNExplainer

In [2]:
graph_dataset_name = "PPI"
embedding_dimension = 64
embedding_method = "GCN"

embeddings_path = f'./Noisy_features/embeddings/{graph_dataset_name}/{embedding_method}_embeddings_{embedding_dimension}.npy'
predictions_path = f'./Noisy_features/predictions/{graph_dataset_name}/{embedding_method}_embeddings_{embedding_dimension}.npy'

graph_file_path = f'data/{graph_dataset_name}/{graph_dataset_name}_combined.txt'
graphLoader = GraphLoader(graph_file_path, graph_dataset_name)
G_nx, G_pyg = graphLoader.load_graph()
num_classes = torch.unique(G_pyg.y).numel()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


Original PPI labels shape: torch.Size([3480, 121])
NetworkX - Number of nodes: 3480
NetworkX - Number of edges: 54806
NetworkX graph is undirected.
NetworkX graph is connected: True
Number of graphs: 6
PyG - Number of features per node: 50
Number of classes: 2
PyG - Number of nodes: 3480
PyG - Number of edges: 108183
Is undirected: True
PyG - Number of features per node: 50


In [3]:
import torch

torch.manual_seed(42)
num_nodes = G_pyg.x.shape[0]
print("Old feature shape:", G_pyg.x.shape)
num_noisy_features = 3
noisy_features = torch.randn(num_nodes, num_noisy_features)
noisy_features = noisy_features - noisy_features.mean(1, keepdim=True)

G_pyg.x = torch.cat([G_pyg.x, noisy_features], dim=1)
print(G_pyg.x)
print("New feature shape:", G_pyg.x.shape)
num_original_features = G_pyg.x.shape[1] - num_noisy_features
noisy_feature_start_index = num_original_features
noisy_feature_indices = list(range(noisy_feature_start_index, noisy_feature_start_index + num_noisy_features))

print("Noisy features start at column index:", noisy_feature_start_index)
print("Noisy features end at column index:", noisy_feature_start_index + num_noisy_features - 1)
print("Indices of noisy features:", noisy_feature_indices)
graph = G_pyg.to(device)

original_feature_names = [f"{i}" for i in range(num_original_features)]
noisy_feature_names = [f"{i}" for i in range(num_original_features, num_original_features + num_noisy_features)]

all_feature_names = original_feature_names + noisy_feature_names

print("All feature names:", all_feature_names)

Old feature shape: torch.Size([3480, 50])
tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.4886,  0.0490, -0.5376],
        [ 0.0000,  0.0000,  0.0000,  ..., -1.2183,  1.5656, -0.3473],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.7569, -0.8047,  0.0478],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ..., -0.9523,  0.8588,  0.0935],
        [ 0.0000,  0.0000,  0.0000,  ..., -0.4831, -0.1584,  0.6416],
        [ 0.0000,  0.0000,  0.0000,  ..., -1.7883,  1.6145,  0.1738]])
New feature shape: torch.Size([3480, 53])
Noisy features start at column index: 50
Noisy features end at column index: 52
Indices of noisy features: [50, 51, 52]
All feature names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52']


In [4]:
y = G_pyg.y
class_freq = y.sum(dim=0)      
freq = class_freq.cpu().numpy()

most_frequent_class = freq.argmax()
most_frequent_value = freq[most_frequent_class]

print(f"Most frequent class: {most_frequent_class}")
print(f"Frequency: {int(most_frequent_value)} nodes")

class_freq = G_pyg.y.sum(dim=0).cpu().numpy()
most_frequent_class = class_freq.argmax()

y_single = G_pyg.y[:, most_frequent_class].float()
print("Training on class:", most_frequent_class)

Most frequent class: 32
Frequency: 3047 nodes
Training on class: 32


In [16]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = torch.nn.BatchNorm1d(hidden_channels)
        self.out = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, return_embeddings=False):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        embeddings = x

        if return_embeddings:
            return embeddings
        logits = self.out(embeddings)
        return logits


graph_dataset_name = "PPI"
embedding_method = "GCN"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = G_pyg.to(device)
y_single = y_single.long().to(device)

num_pos = (y_single == 1).sum().item()
num_neg = (y_single == 0).sum().item()
weights = torch.tensor([1.0, num_neg / num_pos]).to(device)

model = GCN(
    in_channels=data.x.size(1),
    hidden_channels=64
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=20, min_lr=1e-5
)

criterion = torch.nn.CrossEntropyLoss(weight=weights)
def train():
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index)
    loss = criterion(logits[data.train_mask], y_single[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(mask):
    model.eval()
    logits = model(data.x, data.edge_index)
    pred = logits.argmax(dim=1)
    correct = (pred[mask] == y_single[mask]).float().mean()
    return correct.item()


@torch.no_grad()
def evaluate_loss(mask):
    model.eval()
    logits = model(data.x, data.edge_index)
    loss = criterion(logits[mask], y_single[mask])
    return loss.item()

for epoch in range(1, 401):
    loss = train()
    val_loss = evaluate_loss(data.val_mask)

    
    scheduler.step(val_loss)
    if epoch % 20 == 0:
        train_acc = evaluate(data.train_mask)
        val_acc = evaluate(data.val_mask)
        current_lr = optimizer.param_groups[0]['lr']

        print(
            f"Epoch {epoch:03d} | Loss: {loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {current_lr:.5f}"
        )

final_train_acc = evaluate(data.train_mask)
final_val_acc = evaluate(data.val_mask)
final_test_acc = evaluate(data.test_mask)

print("\n=== FINAL ACCURACY REPORT ===")
print(f"Train Accuracy: {final_train_acc:.4f}")
print(f"Val Accuracy:   {final_val_acc:.4f}")
print(f"Test Accuracy:  {final_test_acc:.4f}")

model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    probs = logits.softmax(dim=1)
    prob_pos = probs[:, 1].cpu().numpy()
    pred = logits.argmax(dim=1).cpu().numpy()
    true = y_single.cpu().numpy()
    test_mask_np = data.test_mask.cpu().numpy()

    f1 = f1_score(true[test_mask_np], pred[test_mask_np])
    auc = roc_auc_score(true[test_mask_np], prob_pos[test_mask_np])
    precision = precision_score(true[test_mask_np], pred[test_mask_np])
    recall = recall_score(true[test_mask_np], pred[test_mask_np])

print("\n=== FINAL METRICS ===")
print(f"F1 Score:   {f1:.4f}")
print(f"AUC:        {auc:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")


embeddings = model(data.x, data.edge_index, return_embeddings=True)
embeddings = embeddings.detach().cpu()

predictions_df = pd.DataFrame({
    'Node Index': range(len(pred)),
    'Predicted Class': pred.flatten()
})

print(predictions_df.head())

os.makedirs("Noisy_features_embs", exist_ok=True)

torch.save(
    model.state_dict(),
    f"Noisy_features_embs/{graph_dataset_name}_{embedding_method}.pth"
)

print(f"Model saved to Noisy_features_embs/{graph_dataset_name}_{embedding_method}.pth")


Epoch 020 | Loss: 0.6052 | Train Acc: 0.6193 | Val Acc: 0.6207 | LR: 0.01000
Epoch 040 | Loss: 0.5375 | Train Acc: 0.3502 | Val Acc: 0.3506 | LR: 0.01000
Epoch 060 | Loss: 0.4806 | Train Acc: 0.7385 | Val Acc: 0.6753 | LR: 0.01000
Epoch 080 | Loss: 0.4284 | Train Acc: 0.8276 | Val Acc: 0.7328 | LR: 0.00500
Epoch 100 | Loss: 0.3845 | Train Acc: 0.8549 | Val Acc: 0.7299 | LR: 0.00500
Epoch 120 | Loss: 0.3661 | Train Acc: 0.8452 | Val Acc: 0.6724 | LR: 0.00250
Epoch 140 | Loss: 0.3504 | Train Acc: 0.8772 | Val Acc: 0.7213 | LR: 0.00125
Epoch 160 | Loss: 0.3324 | Train Acc: 0.8847 | Val Acc: 0.7241 | LR: 0.00063
Epoch 180 | Loss: 0.3510 | Train Acc: 0.8689 | Val Acc: 0.7011 | LR: 0.00031
Epoch 200 | Loss: 0.3495 | Train Acc: 0.8865 | Val Acc: 0.7299 | LR: 0.00016
Epoch 220 | Loss: 0.3351 | Train Acc: 0.8811 | Val Acc: 0.7126 | LR: 0.00008
Epoch 240 | Loss: 0.3414 | Train Acc: 0.8779 | Val Acc: 0.7155 | LR: 0.00004
Epoch 260 | Loss: 0.3266 | Train Acc: 0.8861 | Val Acc: 0.7184 | LR: 0.00002

In [6]:
def get_2_hop_neighbors(graph, node):
    one_hop_neighbors = set(graph.neighbors(node))
    two_hop_neighbors = set()
    for neighbor in one_hop_neighbors:
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    return two_hop_neighbors

num_nodes = 200
nodes_with_large_2hop_neighbors = []
for node in G_nx.nodes():
    two_hop_neighbors = get_2_hop_neighbors(G_nx, node)
    if len(two_hop_neighbors) > 100:
        nodes_with_large_2hop_neighbors.append(node)
    if len(nodes_with_large_2hop_neighbors) >= num_nodes:
        break

print(f"Found {len(nodes_with_large_2hop_neighbors)} nodes with 2-hop neighborhoods greater than 100:")
print(len(nodes_with_large_2hop_neighbors))

Found 200 nodes with 2-hop neighborhoods greater than 100:
200


In [7]:
def get_node_features(G, node_attrs, dataset_name, save_dir="Node_Features"):
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{dataset_name}_features.npy")
    columns_path = os.path.join(save_dir, f"{dataset_name}_columns.txt")
    
    
    if os.path.exists(save_path) and os.path.exists(columns_path):
        features = np.load(save_path)
        with open(columns_path, "r") as f:
            columns = f.read().strip().split("\n")
        node_ids = sorted(G.nodes())
        return pd.DataFrame(features, index=node_ids, columns=columns)

  
    feature_columns = [
        "Degree",
        "Avg Neighbor Degree",
        "Clustering Coefficient",
        "Node Triangles",
        "Personalized Pagerank Std",
        "Avg Neighbor Clustering",
    ]
    attr_columns = [f"{str(i)}" for i in range(0,node_attrs.shape[1])]
    all_columns = feature_columns + attr_columns

    features = []
    node_ids = sorted(G.nodes())

    for node in tqdm(node_ids, desc="Computing node features"):
    
        attr = node_attrs[node]
        combined = np.concatenate([attr])
        features.append(combined)

    features = np.array(features)
    np.save(save_path, features)

    with open(columns_path, "w") as f:
        f.write("\n".join(all_columns))

    return pd.DataFrame(features, index=node_ids, columns=all_columns)

### TACENR Approach

In [8]:
from explainer import Explainer

ratios_to_try = [
     (0.8, 0.2)  
]
model_type = "linear"

node_attributes = G_pyg.x
num_of_features = node_attributes.shape[1]


num_nodes = G_pyg.num_nodes
node_attrs = (G_pyg.x.cpu().numpy() if G_pyg.x is not None
                      else np.eye(G_pyg.num_nodes, dtype=np.float32))

feature_names = [str(i) for i in range(0, node_attrs.shape[1])]
node_features_df = get_node_features(G_nx, node_attrs, dataset_name=graph_dataset_name)
N = G_pyg.num_nodes

for sim_ratio, dis_ratio in ratios_to_try:
    print(f"\n--- Testing ratio: {int(sim_ratio*100)}% similar, {int(dis_ratio*100)}% dissimilar ---")
    T = min(int(0.10 * N), 300)
    T = max(T, 20)                      
    T = min(T, N - 1)                    

    m = int(T * sim_ratio)              
    n = T - m                            
    m = max(1, m)
    n = max(1, n)
    mode = "weighted"
    problem = "supervised"
    weighting_method="gradients_only_for_ranking"
    all_pprs = []
    print(f"Using m={m}, n={n} (T={T} ~ {100*T/N:.1f}% of {N})")
    explainer = Explainer(G_nx, G_pyg, node_features_df,all_pprs, nodes_with_large_2hop_neighbors, feature_names, graph_dataset_name, embedding_method, embeddings, model, explanation_type="global", 
                                    node_attributes=True, structural_features = False, problem='regression', model_type=model_type, num_similar=m, num_dissimilar=n, 
                                    scale=True, parameter_tuning=True, enable_plots=False, contrastive=True, weighting_method=weighting_method, include_target_node_features=False, use_proximity= True)
    
    frequency_df, avg_importances_df, all_importances_df, average_metrics, per_node_metrics_df  = explainer.explain(mode = mode, problem = problem)
    


--- Testing ratio: 80% similar, 20% dissimilar ---
Using m=240, n=60 (T=300 ~ 8.6% of 3480)


Processing nodes: 100%|██████████| 200/200 [00:12<00:00, 16.42it/s]


In [ ]:
import os
import csv
output_dir = "Noisy_features_lists"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, f"noisy_counts_TACENR_{embedding_method}_{embedding_dimension}_{graph_dataset_name}.csv")
noisy_counts_TACENR = []
K = 2
for node_id, group in all_importances_df.groupby("Node"):
    top_k = (
        group.reindex(group["Importance"].abs().sort_values(ascending=False).index)
        .head(K)
    )
    count_noisy = top_k["Feature"].astype(str).isin(noisy_feature_names).sum()
    noisy_counts_TACENR.append(count_noisy)

with open(output_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Noisy_Count"])
    for count in noisy_counts_TACENR:
        writer.writerow([count])

### GraphLIME Approach

In [12]:
import networkx as nx

def get_k_hop_neighbors(G, node, k):
    subgraph = nx.ego_graph(G, node, radius=k)
    subgraph_nodes = list(subgraph.nodes())
    subgraph_nodes.sort()
    
    return subgraph_nodes, len(subgraph_nodes)

def get_node_features(data, nodes):
    return data.x[nodes].cpu().numpy()

In [ ]:
from contextlib import redirect_stdout

count = 0
top_feature_counts = Counter()
feature_importances = Counter()
noisy_counts_GraphLIME = []
for node in tqdm(nodes_with_large_2hop_neighbors, desc='Processing nodes'):

    subgraph_nodes, num_nodes =  get_k_hop_neighbors(G_nx, node, 2)
    if num_nodes > 100:
        count+=1
        node_features = get_node_features(G_pyg, subgraph_nodes)
        
        node_features_df = pd.DataFrame(node_features, columns=[f'feature_{i}' for i in range(node_features.shape[1])])
        node_features_df['Node Index'] = subgraph_nodes
        
        merged_df = pd.merge(node_features_df, predictions_df, on='Node Index', how='inner')
        merged_df.reset_index(drop=True, inplace=True)
        
        X = merged_df.drop(columns=['Node Index', 'Predicted Class']).values  
        y = merged_df['Predicted Class'].values
        hsic_lasso = HSICLasso()
        hsic_lasso.input(X, y, M=3, B=16)
        hsic_lasso.classification()

        selected_feature_indexes = hsic_lasso.get_index()
        selected_feature_scores = hsic_lasso.get_index_score()
        top_features = sorted(zip(selected_feature_indexes, selected_feature_scores), key=lambda x: -x[1])[:K]
        num_noisy = sum(1 for feature_idx, _ in top_features if feature_idx in noisy_feature_indices)
        noisy_counts_GraphLIME.append(num_noisy)
        for feature_idx, score in top_features:
            feature_importances[feature_idx] += score

noisy_feature_count_distribution_GraphLIME = pd.Series(Counter(noisy_counts_GraphLIME)).sort_index()


Processing nodes: 100%|██████████| 200/200 [00:40<00:00,  4.98it/s]


In [14]:
output_file = os.path.join(output_dir, f"noisy_counts_GraphLIME_{embedding_method}_{embedding_dimension}_{graph_dataset_name}.csv")
with open(output_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Noisy_Count"])  
    for count in noisy_counts_GraphLIME:
        writer.writerow([count])

### GNNExplainer Approach

In [19]:
explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=50),
    explanation_type='model',
    node_mask_type='attributes',
    edge_mask_type=None,
    model_config=dict(
        mode='multiclass_classification',
        task_level='node',
        return_type='raw',
    ),
)

In [20]:

all_feature_importances = []
noisy_counts_GNNExplainer = []
for node_index in tqdm(nodes_with_large_2hop_neighbors, desc="Explaining nodes"):
    
    explanation = explainer(graph.x, graph.edge_index, index=node_index)
    node_feature_importances = explanation.node_mask
    node_feature_importances = node_feature_importances[node_index].cpu().detach().numpy()
    
    sorted_idx_desc = np.argsort(node_feature_importances)[::-1]
    
    K = 10
    top_k_idx = sorted_idx_desc[:K]
    top_k_idx = [idx for idx in top_k_idx if node_feature_importances[idx] > 0.8] 
    
    num_noisy = sum(1 for idx in top_k_idx if idx in noisy_feature_indices)
    noisy_counts_GNNExplainer.append(num_noisy)

  

Explaining nodes: 100%|██████████| 200/200 [00:50<00:00,  3.94it/s]


In [21]:
output_file = os.path.join(output_dir, f"noisy_counts_GNNExplainer_{embedding_method}_{embedding_dimension}_{graph_dataset_name}.csv")
with open(output_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Noisy_Count"]) 
    for count in noisy_counts_GNNExplainer:
        writer.writerow([count])

In [22]:
results_dict = {
    "TACENR": noisy_counts_TACENR,
    "GraphLIME": noisy_counts_GraphLIME,
    "GNEExplainer": noisy_counts_GNNExplainer
}